## 层和块
 从编程的角度来看，块由类（class）表示。 它的任何子类都必须定义一个将其输入转换为输出的前向传播函数， 并且必须存储任何必需的参数。 注意，有些块不需要任何参数。 最后，为了计算梯度，块必须具有反向传播函数。 在定义我们自己的块时，由于自动微分（在 2.5节 中引入） 提供了一些后端实现，我们只需要考虑前向传播函数和必需的参数。

在构造自定义块之前，我们先回顾一下多层感知机 （ 4.3节 ）的代码。 下面的代码生成一个网络，其中包含一个具有256个单元和ReLU激活函数的全连接隐藏层， 然后是一个具有10个隐藏单元且不带激活函数的全连接输出层。

In [2]:
import torch
from torch import nn
from torch.nn import functional as F

net = nn.Sequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
X = torch.rand(2,20) # 生成2行20列的张量，每个元素都是从[0,1)的均匀分布中独立采样的浮点数
net(X)

tensor([[ 0.0860, -0.0671,  0.1059, -0.0899,  0.0863,  0.1910,  0.1013,  0.0742,
         -0.1315, -0.0271],
        [ 0.1984,  0.0764, -0.0151, -0.1419,  0.0249,  0.1702,  0.0349,  0.2237,
         -0.0642, -0.0478]], grad_fn=<AddmmBackward0>)

在这个例子中，我们通过实例化nn.Sequential来构建我们的模型， 层的执行顺序是作为参数传递的。 简而言之，nn.Sequential定义了一种特殊的Module， 即在PyTorch中表示一个块的类， 它维护了一个由Module组成的有序列表。 注意，两个全连接层都是Linear类的实例， Linear类本身就是Module的子类。 另外，到目前为止，我们一直在通过net(X)调用我们的模型来获得模型的输出。 这实际上是net.__call__(X)的简写。 这个前向传播函数非常简单： 它将列表中的每个块连接在一起，将每个块的输出作为下一个块的输入。

### 自定义块
在下面的代码片段中，我们从零开始编写一个块。 它包含一个多层感知机，其具有256个隐藏单元的隐藏层和一个10维输出层。 注意，下面的MLP类继承了表示块的类。 我们的实现只需要提供我们自己的构造函数（Python中的__init__函数）和前向传播函数。

In [3]:
class MLP(nn.Module):
    # 使用模型参数声明层，这里，我们声明两个全连接的层
    def __init__(self):
        # 调用MLP的父类Moudle的构造函数来执行必要的初始化
        # 这样在类实例化时也可以指定其他函数参数，例如模型参数params
        super().__init__()
        self.hidden = nn.Linear(20,256) # 隐藏层
        self.out = nn.Linear(256,10) # 输出层
    
    # 定义模型的前向传播，即如何根据输入X返回所需的模型输出
    def forward(self,X):
        # 我们使用ReLU函数版本，其在nn.functional模块中定义
        return self.out(F.relu(self.hidden(X)))

我们首先看一下前向传播函数，它以X作为输入， 计算带有激活函数的隐藏表示，并输出其未规范化的输出值。 在这个MLP实现中，两个层都是实例变量。 要了解这为什么是合理的，可以想象实例化两个多层感知机（net1和net2）， 并根据不同的数据对它们进行训练。 当然，我们希望它们学到两种不同的模型。

接着我们实例化多层感知机的层，然后在每次调用前向传播函数时调用这些层。 注意一些关键细节： 首先，我们定制的__init__函数通过super().__init__() 调用父类的__init__函数， 省去了重复编写模版代码的痛苦。 然后，我们实例化两个全连接层， 分别为self.hidden和self.out。 注意，除非我们实现一个新的运算符， 否则我们不必担心反向传播函数或参数初始化， 系统将自动生成这些。

我们来试一下这个函数：

In [4]:
net = MLP()
net(X)

tensor([[-0.0894,  0.2586, -0.1594, -0.2699,  0.0801,  0.1367, -0.1895, -0.0529,
          0.2196, -0.0310],
        [-0.0831,  0.2623, -0.1033, -0.3534,  0.0542,  0.0639, -0.2404, -0.1352,
          0.1447,  0.0571]], grad_fn=<AddmmBackward0>)

块的一个主要优点是它的多功能性。 我们可以子类化块以创建层（如全连接层的类）、 整个模型（如上面的MLP类）或具有中等复杂度的各种组件。 我们在接下来的章节中充分利用了这种多功能性， 比如在处理卷积神经网络时。
### 顺序块
现在我们可以更仔细地看看Sequential类是如何工作的， 回想一下Sequential的设计是为了把其他模块串起来。 为了构建我们自己的简化的MySequential， 我们只需要定义两个关键函数：

1. 一种将块逐个追加到列表中的函数；

2. 一种前向传播函数，用于将输入按追加块的顺序传递给块组成的“链条”。

下面的MySequential类提供了与默认Sequential类相同的功能。

In [5]:
class MySequential(nn.Module):
    def __init__(self,*args):
        super().__init__()  
        for idx,module in enumerate(args):
            # 这里Moudle是Moudle子类的一个实例，我们把它保存在Moudle类的成员变量_modules中。_module的类型是OrderedDict
            self._modules[str(idx)] = module

    def forward(self,X):
        # OrderedDict保证了按照成员添加的顺序遍历他们
        for block in self._modules.values():
            X = block(X)
        return X

__init__函数将每个模块逐个添加到有序字典_modules中。_modules的主要优点是： 在模块的参数初始化过程中， 系统知道在_modules字典中查找需要初始化参数的子块。

当MySequential的前向传播函数被调用时， 每个添加的块都按照它们被添加的顺序执行。 现在可以使用我们的MySequential类重新实现多层感知机。

In [6]:
net = MySequential(nn.Linear(20,256),nn.ReLU(),nn.Linear(256,10))
net(X)

tensor([[ 0.0619,  0.0669,  0.2252,  0.2680, -0.0192, -0.0195, -0.0344,  0.3604,
         -0.0322,  0.2270],
        [ 0.1421,  0.0660,  0.3365,  0.2551, -0.1482, -0.0475, -0.0394,  0.3483,
         -0.0810,  0.3061]], grad_fn=<AddmmBackward0>)

### 在前向传播函数中执行代码
Sequential类使模型构造变得简单， 允许我们组合新的架构，而不必定义自己的类。 然而，并不是所有的架构都是简单的顺序架构。 当需要更强的灵活性时，我们需要定义自己的块。 例如，我们可能希望在前向传播函数中执行Python的控制流。 此外，我们可能希望执行任意的数学运算， 而不是简单地依赖预定义的神经网络层。

到目前为止， 我们网络中的所有操作都对网络的激活值及网络的参数起作用。 然而，有时我们可能希望合并既不是上一层的结果也不是可更新参数的项， 我们称之为常数参数（constant parameter）。例如，我们需要一个计算函数$f(\mathbf{x},\mathbf{w})=c\cdot\mathbf{w}^\top\mathbf{x}$的层，x是输入，w是参数，c是某个在优化过程中没有更新的指定常量。因此我们实现了一个FixedHiddenMLP类，如下所示：

In [9]:
class FixedHiddenMLP(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        # 不计算梯度的随机权重参数，因此其在训练期间保持不变
        self.rand_weight = torch.rand((20,20),requires_grad=False)
        self.linear = nn.Linear(20,20)
    
    def forward(self,X):
        X = self.linear(X)
        # 使用创建的常量参数以及relu和mm函数
        X = F.relu(torch.mm(X,self.rand_weight)+11)
        # 使用全连接层，折相当于两个全连接层共享参数
        X = self.linear(X)  
        # 控制流
        while X.abs().sum() > 1:
            X /= 2
        return X.sum()

在这个FixedHiddenMLP模型中，我们实现了一个隐藏层， 其权重（self.rand_weight）在实例化时被随机初始化，之后为常量。 这个权重不是一个模型参数，因此它永远不会被反向传播更新。 然后，神经网络将这个固定层的输出通过一个全连接层。

注意，在返回输出之前，模型做了一些不寻常的事情： 它运行了一个while循环，在$L_1$范数大于1的条件下，将输出向量除以2，直到它满足条件为止。 最后，模型返回了X中所有项的和。 注意，此操作可能不会常用于在任何实际任务中， 我们只展示如何将任意代码集成到神经网络计算的流程中。

In [10]:
net = FixedHiddenMLP()
net(X)

tensor(-0.0487, grad_fn=<SumBackward0>)

In [11]:
class NestMLP(nn.Module):
    def __init__(self, *args, **kwargs) -> None:
        super().__init__(*args, **kwargs)
        self.net = nn.Sequential(nn.Linear(20,64),nn.ReLU(),nn.Linear(64,32),nn.ReLU())
        self.linear = nn.Linear(32,16) # self.net的输出作为输入
 
    def forward(self,X):
        return self.linear(self.net(X)) 

chimera = nn.Sequential(NestMLP(),nn.Linear(16,20),FixedHiddenMLP())
chimera(X)

tensor(0.1914, grad_fn=<SumBackward0>)

### 小结
- 一个块可以由许多层组成；一个块可以由许多块组成。

- 块可以包含代码。

- 块负责大量的内部处理，包括参数初始化和反向传播。

- 层和块的顺序连接由Sequential块处理。

In [16]:
# 实现一个块，它以两个块为参数，例如net1和net2，并返回前向传播中两个网络的串联输出。这也被称为平行块。
class Parallel(nn.Module):
    def __init__(self, block1, block2) -> None:
        super(Parallel,self).__init__()
        self.block1 = block1    
        self.block2 = block2
    
    def forward(self,X):
        X = torch.cat((self.block1(X),self.block2(X)),1)
        return X

block = Parallel(nn.Linear(20,10),nn.Linear(20,10))
net = nn.Sequential(block,nn.ReLU(),nn.Linear(20,5))
print(net(X).shape)     
        

torch.Size([2, 5])
